# Chapter 5: Adding FAISS to the Chapter 3 Two-Tower Model


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//modern-recommender-systems/blob/main/notebooks/chapter-01/example_notebook.ipynb)

This notebook introduces the basics of recommender systems.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: /Users/kfalk/private/repos/manning/modern-recommender-systems


In [2]:
# Cell 2: Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu


In [3]:

# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

Not running in Colab, skipping setup.
⚠ No GPU detected.


False

In [5]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.data import loaders

DATA_PATH = get_data_path()

In [6]:
print("Ready to build recommender systems!")

Ready to build recommender systems!


The following example shows how to take the item embeddings produced by our Chapter 3 two-tower model and serve them through an FAISS HNSW index for production-scale retrieval.


Assume `two_tower_model` is our trained model from Chapter 3, and `item_dataset` is our full item catalogue.  The retrieval index will extract item embeddings from the trained two-tower model and build an FAISS HNSW index for serving.



index Updates and Maintenance
One operational reality that ANN benchmarks rarely discuss: your catalogue changes constantly. News articles are published every few minutes. Products go out of stock. Videos get taken down. Your ANN index needs a strategy for staying fresh.
There are three common approaches:
Full rebuild (nightly): Rebuild the entire index from scratch on a schedule. Simple and reliable — but your index can be up to 24 hours stale. Acceptable for stable catalogues (e-commerce, movies), problematic for news. Incremental insertion: HNSW supports adding new vectors without full rebuilds. New items can be inserted in near-real-time, though index quality degrades slightly over time and periodic full rebuilds are still recommended. Two-index serving: Maintain a large, stable "base" index rebuilt nightly and a small "fresh" index updated in real-time. At serving time, query both and merge results. This is the most operationally complex but gives you both freshness and quality. For a news recommender, the two-index approach is almost always the right choice. We'll return to the freshness problem when we discuss cold start for new items in Section 5.6.


In [ ]:
import faiss
import numpy as np
import torch
from typing import Tuple

class ANNRetrievalIndex:
    """
    Production-ready ANN retrieval index built on top of
    the two-tower model from Chapter 3.
    
    Supports HNSW (high recall) and IVF-PQ (memory-efficient)
    backends, selectable based on catalogue size and constraints.
    """

    def __init__(
        self,
        embedding_dim: int = 128,
        index_type: str = "hnsw",       # "hnsw" or "ivfpq"
        hnsw_m: int = 32,               # HNSW: num bidirectional links
        ivf_nlist: int = 1024,          # IVF: number of clusters
        pq_m: int = 16,                 # PQ: number of subquantizers
        nprobe: int = 64,               # IVF: clusters to search at query time
    ):
        self.embedding_dim = embedding_dim
        self.index_type = index_type
        self.nprobe = nprobe
        self.item_ids = []              # Maps FAISS integer index → item ID

        if index_type == "hnsw":
            # HNSW index: M controls graph connectivity
            # Higher M → better recall, more memory
            self.index = faiss.IndexHNSWFlat(embedding_dim, hnsw_m)
            self.index.hnsw.efConstruction = 200  # Build quality
            self.index.hnsw.efSearch = 128         # Query quality

        elif index_type == "ivfpq":
            # IVF-PQ: clusters + product quantization
            # Good for memory-constrained billion-scale systems
            quantizer = faiss.IndexFlatIP(embedding_dim)
            self.index = faiss.IndexIVFPQ(
                quantizer,
                embedding_dim,
                ivf_nlist,  # Number of Voronoi cells
                pq_m,       # PQ subquantizers (must divide embedding_dim)
                8            # Bits per subquantizer code
            )
        else:
            raise ValueError(f"Unknown index type: {index_type}")

    def build(
        self,
        item_embeddings: np.ndarray,    # (num_items, embedding_dim)
        item_ids: list,
    ) -> None:
        """
        Build the ANN index from item embeddings.
        Call this offline after training or re-training your two-tower model.
        """
        assert item_embeddings.shape[1] == self.embedding_dim, (
            f"Embedding dim mismatch: expected {self.embedding_dim}, "
            f"got {item_embeddings.shape[1]}"
        )

        # Normalize to unit sphere for cosine similarity via dot product
        faiss.normalize_L2(item_embeddings)

        # IVF indices require a training step to learn cluster centroids
        if self.index_type == "ivfpq":
            print("Training IVF-PQ index (this may take a few minutes)...")
            self.index.train(item_embeddings)
            self.index.nprobe = self.nprobe

        self.index.add(item_embeddings)
        self.item_ids = item_ids
        print(f"Index built: {self.index.ntotal} items indexed.")

    def query(
        self,
        user_embedding: np.ndarray,     # (embedding_dim,) or (batch, embedding_dim)
        k: int = 500,
    ) -> Tuple[list, list]:
        """
        Retrieve top-k candidates for one or more user embeddings.
        
        Returns:
            item_ids:  List of retrieved item IDs (or list of lists for batch)
            scores:    Corresponding similarity scores
        """
        # Ensure 2D input: (batch_size, embedding_dim)
        if user_embedding.ndim == 1:
            user_embedding = user_embedding[np.newaxis, :]

        faiss.normalize_L2(user_embedding)
        scores, indices = self.index.search(user_embedding, k)

        # Map FAISS integer indices back to item IDs
        retrieved_ids = [
            [self.item_ids[idx] for idx in row if idx != -1]
            for row in indices
        ]
        retrieved_scores = [
            [float(s) for s, idx in zip(score_row, idx_row) if idx != -1]
            for score_row, idx_row in zip(scores, indices)
        ]

        # Return single result (not list of lists) for single query
        if len(retrieved_ids) == 1:
            return retrieved_ids[0], retrieved_scores[0]

        return retrieved_ids, retrieved_scores

    def save(self, path: str) -> None:
        """Persist index to disk for serving."""
        faiss.write_index(self.index, f"{path}/faiss.index")
        np.save(f"{path}/item_ids.npy", np.array(self.item_ids))
        print(f"Index saved to {path}")

    def load(self, path: str) -> None:
        """Load a pre-built index from disk."""
        self.index = faiss.read_index(f"{path}/faiss.index")
        self.item_ids = np.load(f"{path}/item_ids.npy").tolist()
        if self.index_type == "ivfpq":
            self.index.nprobe = self.nprobe
        print(f"Index loaded: {self.index.ntotal} items.")


In [ ]:
def build_retrieval_index(two_tower_model, item_dataset, embedding_dim=128):

    two_tower_model.eval()
    all_embeddings = []
    all_item_ids = []

    with torch.no_grad():
        for batch in item_dataset:
            item_ids = batch["item_id"]
            item_features = batch["features"]

            # Forward pass through item tower only
            embeddings = two_tower_model.item_tower(item_features)
            all_embeddings.append(embeddings.cpu().numpy())
            all_item_ids.extend(item_ids)

    item_embeddings = np.vstack(all_embeddings).astype("float32")

    # Build and save the index
    index = ANNRetrievalIndex(embedding_dim=embedding_dim, index_type="hnsw")
    index.build(item_embeddings, all_item_ids)
    index.save("./retrieval_index")

    return index
def serve_recommendations(user_features, two_tower_model, retrieval_index, k=500):
    """
    At serving time: encode the user, query the ANN index.
    """
    two_tower_model.eval()
    with torch.no_grad():
        user_embedding = two_tower_model.user_tower(user_features)
        user_embedding = user_embedding.cpu().numpy().astype("float32")

    candidate_ids, candidate_scores = retrieval_index.query(user_embedding, k=k)
    return candidate_ids, candidate_scores  



Choosing the Right Index
The first decision when building a FAISS index is which index type to use. This choice depends on three factors: the size of your catalogue, how frequently items are added or removed, and how much memory you can afford. The overview in Section 5.2.2 described the three main families. The code below makes that decision explicit through a configuration object.

In [ ]:
@dataclass
class ANNIndexConfig:
    """
    Configuration for FAISS index construction.

    The right configuration depends on your catalogue size,
    update frequency, and memory constraints. The comments
    on each field explain the trade-offs.
    """

    # Dimensionality of your item embeddings.
    # Must match the output dimension of your item tower.
    embedding_dim: int = 128

    # Index type. Three options:
    #
    # "flat":
    #   Exact search. No approximation. Guaranteed to find
    #   the true nearest neighbors every time.
    #   Use when: catalogue < ~100k items, or when you need
    #   a correctness baseline to measure ANN recall against.
    #   Do not use in production at scale — O(N) per query.
    #
    # "hnsw":
    #   Hierarchical Navigable Small World graph.
    #   Excellent recall and query speed. High memory usage
    #   (the graph structure lives in RAM alongside embeddings).
    #   Index construction is slow; incremental updates are
    #   possible but expensive.
    #   Use when: catalogue is relatively stable, memory is
    #   available, and recall is the priority.
    #
    # "ivf":
    #   Inverted File Index. Divides embedding space into
    #   clusters; queries search only the nearest clusters.
    #   Lower memory than HNSW, faster to build, easier to
    #   update. Recall depends on the nprobe parameter.
    #   Use when: catalogue is large (1M+), items are added
    #   frequently, or memory is constrained.
    index_type: str = "hnsw"

    # HNSW-specific parameters.
    #
    # hnsw_m: number of connections per node in the graph.
    # Higher M -> better recall, more memory, slower build.
    # 16-64 is the typical range; 32 is a good default.
    hnsw_m: int = 32

    # hnsw_ef_construction: size of the dynamic candidate list
    # during index construction. Higher -> better recall,
    # slower build. Must be >= hnsw_m. 200 is a good default.
    hnsw_ef_construction: int = 200

    # hnsw_ef_search: size of the dynamic candidate list
    # during querying. Higher -> better recall, slower query.
    # Can be tuned at query time without rebuilding the index.
    # 50-100 is a reasonable starting point.
    hnsw_ef_search: int = 50

    # IVF-specific parameters.
    #
    # ivf_nlist: number of Voronoi cells (clusters).
    # Rule of thumb: sqrt(N) where N is catalogue size.
    # For 1M items: nlist ~ 1000.
    ivf_nlist: int = 1000

    # ivf_nprobe: number of clusters to search at query time.
    # Higher -> better recall, slower query.
    # Typically 5-20% of nlist is a reasonable starting range.
    # This can be tuned at query time.
    ivf_nprobe: int = 50

    # Whether to use GPU for index construction and querying.
    # Requires faiss-gpu. Dramatically speeds up both stages
    # for large catalogues.
    use_gpu: bool = False

The Index Class
With the configuration defined, we can build the index class. The core challenge beyond the raw FAISS calls is identity management: FAISS works with integer indices (0, 1, 2, ...) internally, but your system works with item IDs (article slugs, product SKUs, video identifiers). The index needs to maintain a bidirectional mapping between them.

In [ ]:
class ANNRetrievalIndex:
    """
    Production-ready ANN index for item embedding retrieval.

    Wraps FAISS to provide:
    - Index construction from item embeddings.
    - Fast approximate nearest neighbor querying.
    - Incremental updates (adding new items without full rebuild).
    - Bidirectional mapping between FAISS integer IDs and
      your item identifiers.
    - A correctness baseline via exact search for measuring
      ANN recall during development.

    Typical usage in a news recommender:

        # At startup or after a full catalogue refresh:
        index = ANNRetrievalIndex(config)
        index.build(item_ids, item_embeddings)

        # When a new article is published:
        index.add_items([new_article_id], [new_article_embedding])

        # When a user request arrives:
        user_embedding = user_tower(user_features)
        candidate_ids, scores = index.query(user_embedding, k=500)
    """

    def __init__(self, config: ANNIndexConfig):
        self.config = config
        self.index = None

        # Bidirectional mapping between your item IDs and
        # FAISS's internal integer indices.
        # faiss_id_to_item_id[i] = the item ID at position i
        # item_id_to_faiss_id["article-123"] = i
        self.faiss_id_to_item_id: List[str] = []
        self.item_id_to_faiss_id: Dict[str, int] = {}

    # ----------------------------------------------------------------
    # Index construction
    # ----------------------------------------------------------------

    def _create_empty_index(self) -> faiss.Index:
        """
        Create an empty FAISS index of the configured type.

        Called by build() and can be called again if you need
        to rebuild the index from scratch.
        """
        d = self.config.embedding_dim

        if self.config.index_type == "flat":
            # Exact search. No parameters to tune.
            # IndexFlatIP uses inner product (dot product) similarity.
            # Use IndexFlatL2 if your embeddings are not normalized
            # and you want Euclidean distance instead.
            index = faiss.IndexFlatIP(d)

        elif self.config.index_type == "hnsw":
            # HNSW graph index.
            # M is the number of connections per node.
            index = faiss.IndexHNSWFlat(d, self.config.hnsw_m)

            # ef_construction controls recall during build.
            # Must be set before adding any vectors.
            index.hnsw.efConstruction = self.config.hnsw_ef_construction

            # ef_search controls recall during querying.
            # Can be changed at any time without rebuilding.
            index.hnsw.efSearch = self.config.hnsw_ef_search

        elif self.config.index_type == "ivf":
            # IVF index. Requires a quantizer — a flat index
            # that maps embeddings to their nearest cluster centroid.
            quantizer = faiss.IndexFlatIP(d)
            index = faiss.IndexIVFFlat(
                quantizer,
                d,
                self.config.ivf_nlist,
                faiss.METRIC_INNER_PRODUCT,
            )
            # nprobe can be changed at query time.
            index.nprobe = self.config.ivf_nprobe

        else:
            raise ValueError(
                f"Unknown index type: {self.config.index_type}. "
                f"Expected one of: flat, hnsw, ivf."
            )

        # Move to GPU if configured and available.
        if self.config.use_gpu:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            logger.info("FAISS index moved to GPU.")

        return index

    def build(
        self,
        item_ids: List[str],
        item_embeddings: np.ndarray,
    ) -> None:
        """
        Build the index from scratch over a full catalogue.

        Call this at startup or after a full catalogue refresh.
        For incremental updates (new items arriving continuously),
        use add_items() after the initial build.

        Args:
            item_ids:
                List of item identifiers, one per embedding.
                These are the IDs your system uses — article
                slugs, product SKUs, video IDs, etc.
            item_embeddings:
                Float32 array of shape (num_items, embedding_dim).
                Embeddings should be L2-normalized if you are
                using inner product similarity (recommended —
                normalized inner product equals cosine similarity).
        """
        assert len(item_ids) == len(item_embeddings), (
            f"Number of item IDs ({len(item_ids)}) must match "
            f"number of embeddings ({len(item_embeddings)})."
        )
        assert item_embeddings.shape[1] == self.config.embedding_dim, (
            f"Embedding dimension mismatch. "
            f"Config expects {self.config.embedding_dim}, "
            f"got {item_embeddings.shape[1]}."
        )

        # FAISS requires float32. Cast if necessary.
        embeddings = item_embeddings.astype(np.float32)

        # L2 normalize so inner product = cosine similarity.
        # If your item tower already outputs normalized embeddings
        # (which it should — see Chapter 3), this is a no-op.
        faiss.normalize_L2(embeddings)

        # Build identity mapping.
        self.faiss_id_to_item_id = list(item_ids)
        self.item_id_to_faiss_id = {
            item_id: i for i, item_id in enumerate(item_ids)
        }

        # Create and populate the index.
        self.index = self._create_empty_index()

        # IVF indices must be trained on a representative sample
        # of the data before vectors can be added. HNSW and Flat
        # indices do not require a training step.
        if self.config.index_type == "ivf":
            logger.info(
                f"Training IVF index on {len(embeddings)} vectors "
                f"with nlist={self.config.ivf_nlist}..."
            )
            self.index.train(embeddings)

        self.index.add(embeddings)

        logger.info(
            f"Built {self.config.index_type} index with "
            f"{self.index.ntotal} vectors."
        )

    # ----------------------------------------------------------------
    # Querying
    # ----------------------------------------------------------------

    def query(
        self,
        query_embedding: np.ndarray,
        k: int = 500,
    ) -> Tuple[List[str], List[float]]:
        """
        Find the k nearest items to a query embedding.

        This is the method called on every user request.
        It should return in under 10ms for catalogues up to
        a few million items with a well-configured index.

        Args:
            query_embedding:
                Float32 array of shape (1, embedding_dim) or
                (embedding_dim,). The user embedding produced
                by your user tower.
            k:
                Number of candidates to return. This is the
                size of the candidate pool passed to your
                scoring and ranking stages. Typical values:
                200-1000 depending on downstream pipeline.

        Returns:
            Tuple of:
            - candidate_ids: list of k item IDs, ordered by
              descending similarity score.
            - scores: corresponding similarity scores.
        """
        if self.index is None:
            raise RuntimeError(
                "Index has not been built. Call build() first."
            )

        # Ensure correct shape and dtype.
        embedding = np.array(query_embedding, dtype=np.float32)
        if embedding.ndim == 1:
            embedding = embedding.reshape(1, -1)

        # Normalize the query embedding.
        # Consistent with how item embeddings were normalized
        # during index construction.
        faiss.normalize_L2(embedding)

        # Search the index.
        # scores: (1, k) array of similarity scores.
        # faiss_ids: (1, k) array of FAISS integer indices.
        scores, faiss_ids = self.index.search(embedding, k)

        # Convert FAISS integer indices back to item IDs.
        # FAISS returns -1 for positions it cannot fill
        # (e.g. if k > index size). Filter these out.
        candidate_ids = []
        candidate_scores = []

        for faiss_id, score in zip(faiss_ids[0], scores[0]):
            if faiss_id == -1:
                # Index has fewer items than k. Stop here.
                break
            candidate_ids.append(
                self.faiss_id_to_item_id[faiss_id]
            )
            candidate_scores.append(float(score))

        return candidate_ids, candidate_scores

    def batch_query(
        self,
        query_embeddings: np.ndarray,
        k: int = 500,
    ) -> List[Tuple[List[str], List[float]]]:
        """
        Query the index for multiple users simultaneously.

        More efficient than calling query() in a loop when
        you need to generate recommendations for multiple
        users at once — for example, during batch offline
        recommendation generation.

        Args:
            query_embeddings:
                Float32 array of shape (num_queries, embedding_dim).
            k:
                Number of candidates per query.

        Returns:
            List of (candidate_ids, scores) tuples, one per query.
        """
        if self.index is None:
            raise RuntimeError(
                "Index has not been built. Call build() first."
            )

        embeddings = np.array(query_embeddings, dtype=np.float32)
        faiss.normalize_L2(embeddings)

        scores, faiss_ids = self.index.search(embeddings, k)

        results = []
        for query_scores, query_faiss_ids in zip(scores, faiss_ids):
            candidate_ids = []
            candidate_scores = []
            for faiss_id, score in zip(query_faiss_ids, query_scores):
                if faiss_id == -1:
                    break
                candidate_ids.append(
                    self.faiss_id_to_item_id[faiss_id]
                )
                candidate_scores.append(float(score))
            results.append((candidate_ids, candidate_scores))

        return results

    # ----------------------------------------------------------------
    # Incremental updates
    # ----------------------------------------------------------------

    def add_items(
        self,
        item_ids: List[str],
        item_embeddings:

In [ ]:
import numpy as np
import faiss
import torch
import torch.nn.functional as F
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
import logging

logger = logging.getLogger(__name__)